# NeRF 튜토리얼

이 노트북은 AI Expert 강의를 위해 제작되었습니다.
이 튜토리얼에서는 NeRF의 기초와 응용을 살펴봅니다.

참고: NeRF [NeRF: Representing Scenes as Neural Radiance Fields for View Synthesis](https://arxiv.org/abs/2003.08934).

<img src="assets/nerf_teaser.png" width=720>

## 목차
* TinyNeRF
* NeRFStudio

# 1. TinyNeRF

빠른 학습과 시각화를 위해 성능을 낮춘 NeRF 모델입니다.
* 원본 NeRF 대비 파라미터 수가 약 20배 적습니다
* 5D 입력에 view direction을 포함하지 않습니다
* Hierarchical Sampling을 수행하지 않습니다

<img src="assets/nerf_pipeline.png" width=720>

In [ ]:
import os,sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# /data is the shared data dir (bind-mounted into the container; also a symlink
# to <repo>/data on the host), the same place nerfstudio scenes live.
if not os.path.exists('/data/tiny_nerf_data.npz'):
    !wget http://cseweb.ucsd.edu/~viscomp/projects/LF/papers/ECCV20/nerf/tiny_nerf_data.npz -O /data/tiny_nerf_data.npz

In [ ]:
#Search for GPU to run on
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#Load in data
rawData = np.load("/data/tiny_nerf_data.npz")
images = rawData["images"]
poses = rawData["poses"]
focal = rawData["focal"]
H, W = images.shape[1:3]
H = int(H)
W = int(W)
print(images.shape, poses.shape, focal)

testimg, testpose = images[99], poses[99]
plt.imshow(testimg)
plt.show()
images = torch.Tensor(images).to(device)
poses = torch.Tensor(poses).to(device)
testimg = torch.Tensor(testimg).to(device)
testpose = torch.Tensor(testpose).to(device)

In [ ]:
def get_rays(H, W, focal, pose):
  i, j = torch.meshgrid(
      torch.arange(W, dtype=torch.float32),
      torch.arange(H, dtype=torch.float32)
      )
  i = i.t()
  j = j.t()
  dirs = torch.stack(
      [(i-W*0.5)/focal,
       -(j-H*0.5)/focal,
       -torch.ones_like(i)], -1).to(device)
  rays_d = torch.sum(dirs[..., np.newaxis, :] * pose[:3, :3], -1)
  rays_o = pose[:3,-1].expand(rays_d.shape)
  return rays_o, rays_d

### NeRF 핵심 수식

아래 `positional_encoder` 와 `render` 가 구현하는 두 핵심 식입니다.

**Positional Encoding** — 좌표를 여러 주파수의 sin/cos로 매핑해 고주파 디테일을 표현합니다:

$$\gamma(p)=\big(\sin(2^0\pi p),\cos(2^0\pi p),\dots,\sin(2^{L-1}\pi p),\cos(2^{L-1}\pi p)\big)$$

**Volume Rendering** — ray $\mathbf{r}(t)=\mathbf{o}+t\mathbf{d}$ 를 따라 색과 밀도를 적분해 픽셀 색을 얻습니다:

$$C(\mathbf{r})=\int_{t_n}^{t_f} T(t)\,\sigma(\mathbf{r}(t))\,\mathbf{c}(\mathbf{r}(t),\mathbf{d})\,dt,\quad T(t)=\exp\!\Big(-\!\int_{t_n}^{t}\sigma(\mathbf{r}(s))\,ds\Big)$$

여기서 $\sigma$ 는 밀도(density), $\mathbf{c}$ 는 색(color), $T(t)$ 는 누적 투과도(accumulated transmittance)입니다.
구현에서는 이 적분을 ray 위 $N$ 개 샘플로 이산화하여 계산합니다.

In [ ]:
def positional_encoder(x, L_embed=6):
  rets = [x]
  for i in range(L_embed):
    for fn in [torch.sin, torch.cos]:
      rets.append(fn(2.**i *x))#(2^i)*x
  return torch.cat(rets, -1)

def cumprod_exclusive(tensor: torch.Tensor) -> torch.Tensor:
  cumprod = torch.cumprod(tensor, -1)
  cumprod = torch.roll(cumprod, 1, -1)
  cumprod[..., 0] = 1.
  return cumprod

def render(model, rays_o, rays_d, near, far, n_samples, rand=False):
  def batchify(fn, chunk=1024*32):
      return lambda inputs: torch.cat([fn(inputs[i:i+chunk]) for i in range(0, inputs.shape[0], chunk)], 0)

  z = torch.linspace(near, far, n_samples).to(device)
  if rand:
    mids = 0.5 * (z[..., 1:] + z[...,:-1])
    upper = torch.cat([mids, z[...,-1:]], -1)
    lower = torch.cat([z[...,:1], mids], -1)
    t_rand = torch.rand(z.shape).to(device)
    z = lower + (upper-lower)*t_rand

  points = rays_o[..., None,:] + rays_d[..., None,:] * z[...,:,None]

  flat_points = torch.reshape(points, [-1, points.shape[-1]])
  flat_points = positional_encoder(flat_points)
  raw = batchify(model)(flat_points)
  raw = torch.reshape(raw, list(points.shape[:-1]) + [4])

  #Compute opacitices and color
  sigma = F.relu(raw[..., 3])
  rgb = torch.sigmoid(raw[..., :3])

  #Volume Rendering
  one_e_10 = torch.tensor([1e10], dtype=rays_o.dtype).to(device)
  dists = torch.cat((z[..., 1:] - z[..., :-1],
                  one_e_10.expand(z[..., :1].shape)), dim=-1)
  alpha = 1. - torch.exp(-sigma * dists)
  weights = alpha * cumprod_exclusive(1. - alpha + 1e-10)

  rgb_map = (weights[...,None]* rgb).sum(dim=-2)
  depth_map = (weights * z).sum(dim=-1)
  acc_map = weights.sum(dim=-1)
  return rgb_map, depth_map, acc_map


In [ ]:
#helper functions
mse2psnr = lambda x : -10. * torch.log(x) / torch.log(torch.Tensor([10.])).to(device)

def train(model, optimizer, n_iters = 3001):
  #Track loss over time for graphing
  psnrs = []
  iternums = []
  plot_step = 500
  n_samples = 64
  for i in range(n_iters):
    #Choose random image and use it for training
    images_idx = np.random.randint(images.shape[0])
    target = images[images_idx]
    pose = poses[images_idx]

    #Core optimizer loop
    rays_o, rays_d = get_rays(H, W, focal, pose)
    rgb, disp, acc = render(model, rays_o, rays_d, near=2., far=6., n_samples=n_samples, rand=True)
    optimizer.zero_grad()
    image_loss = torch.nn.functional.mse_loss(rgb, target)
    image_loss.backward()
    optimizer.step()

    if i%plot_step==0:
      #Render shown image above as model begins to learn
      with torch.no_grad():
        rays_o, rays_d = get_rays(H, W, focal, testpose)
        rgb, depth, acc = render(model, rays_o, rays_d, near=2., far=6., n_samples=n_samples)
        loss = torch.nn.functional.mse_loss(rgb, testimg)
        psnr = mse2psnr(loss).cpu()

        psnrs.append(psnr)
        iternums.append(i)

        plt.figure(figsize=(10,5))
        plt.subplot(121)
        #copy from gpu memory to cpu
        picture = rgb.cpu()
        plt.imshow(picture)
        plt.title(f'Iterations: {i}')
        plt.subplot(122)
        plt.plot(iternums, psnrs)
        plt.title('PSNR')
        plt.show()

In [ ]:
class VeryTinyNerfModel(torch.nn.Module):
  def __init__(self, filter_size=128, num_encoding_functions=6):
    super(VeryTinyNerfModel, self).__init__()
    # Input layer (default: 39 -> 128)
    self.layer1 = torch.nn.Linear(3 + 3 * 2 * num_encoding_functions, filter_size)
    # Layer 2 (default: 128 -> 128)
    self.layer2 = torch.nn.Linear(filter_size, filter_size)
    # Layer 3 (default: 128 -> 4)
    self.layer3 = torch.nn.Linear(filter_size, 4)
    # Short hand for torch.nn.functional.relu
    self.relu = torch.nn.functional.relu

  def forward(self, x):
    x = self.relu(self.layer1(x))
    x = self.relu(self.layer2(x))
    x = self.layer3(x)
    return x

In [ ]:
#Run all the actual code
nerf = VeryTinyNerfModel()
nerf = nn.DataParallel(nerf).to(device)
optimizer = torch.optim.Adam(nerf.parameters(), lr=5e-3, eps = 1e-7)
train(nerf, optimizer)

In [ ]:
%matplotlib inline
from ipywidgets import interactive, widgets


trans_t = lambda t : torch.tensor([
    [1,0,0,0],
    [0,1,0,0],
    [0,0,1,t],
    [0,0,0,1],
], dtype=torch.float32)

rot_phi = lambda phi : torch.tensor([
    [1,0,0,0],
    [0,np.cos(phi),-np.sin(phi),0],
    [0,np.sin(phi), np.cos(phi),0],
    [0,0,0,1],
], dtype=torch.float32)

rot_theta = lambda th : torch.tensor([
    [np.cos(th),0,-np.sin(th),0],
    [0,1,0,0],
    [np.sin(th),0, np.cos(th),0],
    [0,0,0,1],
], dtype=torch.float32)


def pose_spherical(theta, phi, radius):
    c2w = trans_t(radius)
    c2w = rot_phi(phi/180.*np.pi) @ c2w
    c2w = rot_theta(theta/180.*np.pi) @ c2w
    c2w = torch.tensor([[-1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]], dtype=torch.float32) @ c2w
    return c2w


def f(**kwargs):
    c2w = pose_spherical(**kwargs).cuda()
    rays_o, rays_d = get_rays(H, W, focal, c2w[:3,:4])
    rgb, depth, acc = render(nerf, rays_o, rays_d, near=2., far=6., n_samples=64)
    img = np.clip(rgb.cpu().detach().numpy(),0,1)

    plt.figure(2, figsize=(20,6))
    plt.imshow(img)
    plt.show()


sldr = lambda v, mi, ma: widgets.FloatSlider(
    value=v,
    min=mi,
    max=ma,
    step=.01,
)

names = [
    ['theta', [100., 0., 360]],
    ['phi', [-30., -90, 0]],
    ['radius', [4., 3., 5.]],
]

interactive_plot = interactive(f, **{s[0] : sldr(*s[1]) for s in names})
output = interactive_plot.children[-1]
output.layout.height = '350px'
interactive_plot

#### Exercise 6: Custom TinyNeRF

직접 자신만의 TinyNeRF 모델을 설계해봅니다. 아래 코드 셀의 `CustomTinyNerfModel`을 채워주세요.

**호환 조건** (`positional_encoder` / `render` / `train`과 그대로 동작하려면)
- `forward()` 입력: `(N, 39)` — positional encoding된 좌표 (`3 + 3*2*L`, `L=6`)
- `forward()` 출력: `(N, 4)` — `[..., :3]` = RGB, `[..., 3]` = 밀도(sigma)

**시도해볼 수 있는 기법** (참고 논문: NeRF, Mildenhall et al., ECCV 2020)
- MLP를 더 깊게/넓게 만들기 (layer 수, hidden size 늘리기)
- **Skip connection**: 중간 레이어에서 입력 인코딩(`x`)을 다시 이어붙이기
- **density / color head 분리**: 마지막에 sigma용·RGB용 `Linear`를 따로 두기

> 💡 깊은 네트워크는 baseline(2-layer)보다 학습이 민감합니다. learning rate를 `5e-3` 대신 `1e-3` 정도로 낮춰보세요.

In [ ]:
class CustomTinyNerfModel(torch.nn.Module):
    """Exercise 6 (reference answer): an upgraded TinyNeRF.

    `VeryTinyNerfModel` is only 2 hidden layers with no skip connection. Here we
    add two well-established ideas from the NeRF literature, while staying fully
    compatible with `positional_encoder` / `render`
    (input: 39-D encoded point, output: 4-D = [rgb(3), sigma(1)]):

      1. Deeper MLP with a SKIP CONNECTION that re-injects the positional
         encoding at a middle layer, so high-frequency input coordinates are
         not washed out in the deeper layers.
         -> NeRF (Mildenhall et al., ECCV 2020), the Fig.7 8-layer architecture.
      2. SEPARATE density / color heads: sigma and RGB are predicted by their
         own final linear layers from the shared trunk feature, following the
         same paper's head design.

    Note: original NeRF additionally feeds the view direction into the color
    head for view-dependent effects. TinyNeRF drops view direction (5D input
    only), so we keep a single shared trunk here.

    On tiny_nerf_data this reaches ~25.4 PSNR @3001 iters vs ~23.8 for the
    2-layer baseline (deeper net + skip, lr 1e-3).
    """
    def __init__(self, num_encoding_functions=6, filter_size=256,
                 num_layers=6, skip=(3,)):
        super().__init__()
        in_dim = 3 + 3 * 2 * num_encoding_functions   # 39 when L=6
        self.skip = set(skip)
        self.layers = torch.nn.ModuleList()
        for i in range(num_layers):
            if i == 0:
                self.layers.append(torch.nn.Linear(in_dim, filter_size))
            elif i in self.skip:
                # skip layer takes [hidden ; input encoding] concatenated
                self.layers.append(torch.nn.Linear(filter_size + in_dim, filter_size))
            else:
                self.layers.append(torch.nn.Linear(filter_size, filter_size))
        self.sigma_head = torch.nn.Linear(filter_size, 1)   # density
        self.rgb_head   = torch.nn.Linear(filter_size, 3)   # color
        self.relu = torch.nn.functional.relu

    def forward(self, x):
        h = x
        for i, layer in enumerate(self.layers):
            if i in self.skip:
                h = torch.cat([h, x], dim=-1)   # re-inject input positional encoding
            h = self.relu(layer(h))
        sigma = self.sigma_head(h)
        rgb   = self.rgb_head(h)
        # render() expects [..., :3] = RGB, [..., 3] = density
        return torch.cat([rgb, sigma], dim=-1)

nerf = CustomTinyNerfModel()
nerf = nn.DataParallel(nerf).to(device)
# A deeper network is more sensitive: use a smaller lr than the 2-layer baseline (5e-3).
optimizer = torch.optim.Adam(nerf.parameters(), lr=1e-3, eps = 1e-7)
train(nerf, optimizer)

# 2. NeRFStudio

다양한 NeRF 모델을 손쉽게 활용할 수 있도록 설계된 플랫폼입니다.

* 다양한 종류의 NeRF 모델을 포함합니다 (dynamic nerf, editing nerf, 3d diffusion model, fast nerf)
* 원하는 view 이미지를 쉽게 렌더링할 수 있는 강력한 visualizer를 지원합니다
* dataloader, ray sampler, encoder 등 NeRF 모듈을 쉽게 수정할 수 있어 새로운 모델 개발이 용이합니다

### Blender (NeRF synthetic) 데이터 학습

다른 씬을 쓰려면 `--data /data/blender/<scene>` 의 `<scene>` 만 바꾸면 됩니다:
`chair`, `drums`, `ficus`, `hotdog`, `lego`, `materials`, `mic`, `ship`

In [ ]:
# Blender (NeRF synthetic) data — use the blender-data parser
NERFSTUDIO_PORT = os.environ.get("NERFSTUDIO_PORT", "7007")
!ns-train nerfacto --output-dir "results/nerfstudio/nerfacto" \
    --viewer.websocket-port "{NERFSTUDIO_PORT}" \
    blender-data --data /data/blender/lego

### mip-NeRF 360 데이터 학습

다른 씬을 쓰려면 `--data /data/mipnerf360/<scene>` 의 `<scene>` 만 바꾸면 됩니다:
`bicycle`, `bonsai`, `counter`, `flowers`, `garden`, `kitchen`, `room`, `stump`, `treehill`

해상도는 `--downscale-factor` 를 `2` / `4` / `8` 중에서 고르면 됩니다 (각각 `images_2/4/8`).
속도를 위해 **1/8 해상도(`images_8`)** 로 학습하는 것을 권장합니다 (`--downscale-factor 8`).

In [ ]:
# mip-NeRF 360 data — colmap parser at 1/8 resolution (images_8)
NERFSTUDIO_PORT = os.environ.get("NERFSTUDIO_PORT", "7007")
!ns-train nerfacto --output-dir "results/nerfstudio/nerfacto" \
    --viewer.websocket-port "{NERFSTUDIO_PORT}" \
    colmap --data /data/mipnerf360/garden --colmap-path sparse/0 --downscale-factor 8

In [ ]:
# Resume a stopped run — load the checkpoint from the most recent run.
# Note: pass the SAME dataparser / data args as the original run.
#       (Blender example below; for mip-NeRF 360, switch to the colmap args.)
import glob

runs = sorted(glob.glob("results/nerfstudio/nerfacto/*/*/*/nerfstudio_models"))
latest = runs[-1]
print("resuming from:", latest)

NERFSTUDIO_PORT = os.environ.get("NERFSTUDIO_PORT", "7007")
!ns-train nerfacto --load-dir "{latest}" \
    --output-dir "results/nerfstudio/nerfacto" \
    --viewer.websocket-port "{NERFSTUDIO_PORT}" \
    blender-data --data /data/blender/lego

### NeRFStudio로 다양한 모델 활용하기
* nerfacto는 여러 논문의 구성 요소를 조합하여 만든 파이프라인입니다
* nerfstudio는 nerfacto 외에도 다양한 모델을 지원합니다
  - Instant-NGP
  - [Instruct-NeRF2NeRF](https://docs.nerf.studio/en/latest/nerfology/methods/in2n.html)
  - K-Planes
  - [LERF](https://docs.nerf.studio/en/latest/nerfology/methods/lerf.html)
  - Mip-NeRF
  - NeRF
  - Nerfacto
  - Nerfbusters
  - NeRFPlayer
  - Tetra-NeRF
  - TensoRF
  - [Generfacto](https://docs.nerf.studio/en/latest/nerfology/methods/generfacto.html)

In [ ]:
!ns-train --help

#### Exercise 7: NeRFacto 외의 다른 모델 학습해보기 (추천: Instant-NGP)

In [ ]:
######## Implement from here ########
####### End of Implementation #######